# 01 - Connecting to Instruments with PyVISA

This notebook shows a practical first step in RF and instrument automation: **finding instruments, opening a VISA session, and querying `*IDN?`**.

## What this notebook covers

- Creating a VISA resource manager
- Listing available instrument resources
- Opening a selected resource
- Configuring timeout and termination characters
- Sending a standard SCPI identification query
- Handling communication failures
- Using a mock instrument when real hardware is unavailable

## VISA vs. SCPI

- **VISA** is the communication layer used to discover and connect to instruments.
- **SCPI** is the command language used once the connection is open.

A common first command is `*IDN?`, which asks the instrument to identify itself.

In [ ]:
import pyvisa
from pprint import pprint

try:
    rm = pyvisa.ResourceManager()
    print('VISA resource manager created successfully.')
except Exception as e:
    print('Failed to create VISA resource manager:')
    print(e)

## List available resources

Depending on your setup, you may see USB, TCPIP, serial, or GPIB resources.

Examples:
- `TCPIP0::192.168.1.100::inst0::INSTR`
- `USB0::0x0957::0x1798::MY12345678::INSTR`
- `GPIB0::18::INSTR`

In [ ]:
resources = ()

try:
    resources = rm.list_resources()
    print('Discovered VISA resources:')
    pprint(resources)
except Exception as e:
    print('Could not list VISA resources:')
    print(e)

## Choose a resource

For a first tutorial, the simplest method is to select the first discovered resource.

In real projects, you would usually:
- select by known resource string,
- search for a particular vendor or model from `*IDN?`, or
- load addresses from a configuration file.

In [ ]:
selected_resource = resources[0] if len(resources) > 0 else None
print('Selected resource:', selected_resource)

## Open the resource and query `*IDN?`

The code below sets a timeout and line termination characters before issuing a query. This often resolves the most common first-connection issues.

In [ ]:
inst = None

if selected_resource is None:
    print('No VISA resources were found. Skipping real instrument connection.')
else:
    try:
        inst = rm.open_resource(selected_resource)
        inst.timeout = 5000  # milliseconds
        inst.write_termination = '\n'
        inst.read_termination = '\n'

        print(f'Opened: {selected_resource}')
        idn = inst.query('*IDN?')
        print('Instrument identification:')
        print(idn)
    except Exception as e:
        print('Failed to open or query the instrument:')
        print(e)

## A slightly cleaner helper function

As projects grow, it is helpful to wrap repeated connection logic in a utility function.

In [ ]:
def connect_and_identify(resource_name, timeout_ms=5000):
    """Open a VISA resource and return its *IDN? response."""
    inst = rm.open_resource(resource_name)
    inst.timeout = timeout_ms
    inst.write_termination = '\n'
    inst.read_termination = '\n'
    idn = inst.query('*IDN?').strip()
    return inst, idn

In [ ]:
if selected_resource is not None:
    try:
        test_inst, test_idn = connect_and_identify(selected_resource)
        print('Connected successfully.')
        print('IDN:', test_idn)
    except Exception as e:
        print('connect_and_identify failed:')
        print(e)

## Mock instrument mode

This section is useful when you do not have hardware connected.

In [ ]:
class MockInstrument:
    def __init__(self):
        self.timeout = 5000
        self.write_termination = '\n'
        self.read_termination = '\n'
        self.command_log = []

    def write(self, command):
        self.command_log.append(('write', command))

    def query(self, command):
        self.command_log.append(('query', command))
        if command.strip() == '*IDN?':
            return 'MOCK,RF-DEMO,123456,1.0'
        return '0'

    def close(self):
        self.command_log.append(('close', None))

In [ ]:
mock = MockInstrument()
print(mock.query('*IDN?'))
print(mock.command_log)

## Common failure modes

When `*IDN?` does not work, the usual causes are:

1. Wrong VISA resource string
2. Incorrect termination characters
3. Timeout too short
4. Wrong interface selection on the instrument
5. Vendor-specific setup requirements
6. Missing VISA backend or driver installation

## Next step

The natural follow-up notebook is controlling a signal generator:

- set frequency,
- set output power,
- enable RF output safely,
- and confirm settings with query commands.